# ComfyUI for InfiniteTalk (Wan2.1) on Colab

This notebook sets up a ComfyUI environment optimized for **Tesla T4 (16GB)** usage with **InfiniteTalk**.

It includes:
- **ComfyUI-WanVideoWrapper** (Kijai)
- **InfiniteTalk Custom Nodes** (MeiGen-AI)
- **FP8 Model Support** (Critical for 16GB VRAM)
- **Cloudflared Tunnel** for public access

In [6]:
import os
from pathlib import Path

# 1. Install ComfyUI
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI
    %cd ComfyUI
    !pip install -r requirements.txt
else:
    %cd ComfyUI

# 2. Install Manager & Custom Nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager custom_nodes/ComfyUI-Manager
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper custom_nodes/ComfyUI-WanVideoWrapper
!git clone -b comfyui https://github.com/MeiGen-AI/InfiniteTalk custom_nodes/ComfyUI-InfiniteTalk
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite custom_nodes/ComfyUI-VideoHelperSuite
!git clone https://github.com/kijai/ComfyUI-KJNodes custom_nodes/ComfyUI-KJNodes
!git clone https://github.com/christian-byrne/audio-separation-nodes-comfyui custom_nodes/audio-separation-nodes-comfyui

# Install dependencies for custom nodes
!pip install -r custom_nodes/ComfyUI-WanVideoWrapper/requirements.txt
!pip install -r custom_nodes/ComfyUI-InfiniteTalk/requirements.txt
!pip install -r custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt
!pip install -r custom_nodes/ComfyUI-KJNodes/requirements.txt
!pip install -r custom_nodes/audio-separation-nodes-comfyui/requirements.txt

print("Skipping FlashAttention installation.")

Cloning into 'ComfyUI'...
remote: Enumerating objects: 65619, done.
remote: Counting objects: 100% (1188/1188), done.
remote: Compressing objects: 100% (443/443), done.
remote: Total 65619 (delta 945), reused 832 (delta 744), pack-reused 64431 (from 4)
Receiving objects: 100% (65619/65619), 106.45 MiB | 14.48 MiB/s, done.
Resolving deltas: 100% (45383/45383), done.
/content/ComfyUI/ComfyUI
Cloning into 'custom_nodes/ComfyUI-Manager'...
remote: Enumerating objects: 33935, done.
remote: Counting objects: 100% (917/917), done.
remote: Compressing objects: 100% (433/433), done.
remote: Total 33935 (delta 731), reused 552 (delta 484), pack-reused 33018 (from 3)
Receiving objects: 100% (33935/33935), 199.63 MiB | 24.08 MiB/s, done.
Resolving deltas: 100% (25054/25054), done.
Cloning into 'custom_nodes/ComfyUI-WanVideoWrapper'...
remote: Enumerating objects: 6675, done.
remote: Counting objects: 100% (3627/3627), done.
remote: Compressing objects: 100% (276/276), done.
remote: Total 6675 (del

In [7]:
from pathlib import Path

checks = {
    "ComfyUI": "/content/ComfyUI/main.py",
    "InfiniteTalk nodes": "/content/ComfyUI/custom_nodes/ComfyUI-InfiniteTalk",
    "WanVideoWrapper": "/content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper",
    "Downloaded models": "/content/models/checkpoints/infinitetalk_multi_fp8.safetensors"
}

for name, path in checks.items():
    print(("✅ " if Path(path).exists() else "❌ ") + name)

✅ ComfyUI
✅ InfiniteTalk nodes
✅ WanVideoWrapper
✅ Downloaded models


In [9]:
from pathlib import Path

source = Path("/content/models")
destination = Path("/content/ComfyUI/models")

folders = [
    "diffusion_models",
    "text_encoders",
    "vae",
    "clip_vision",
    "checkpoints",
    "loras"
]

for folder in folders:
    src = source / folder
    dst = destination / folder

    dst.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        print("⚠️ Missing:", folder)
        continue

    for file in src.iterdir():
        target = dst / file.name

        if file.is_file() and not target.exists() and not target.is_symlink():
            target.symlink_to(file.resolve())

    print("✅ Connected:", folder)

print("\nModel linking completed!")

✅ Connected: diffusion_models
✅ Connected: text_encoders
✅ Connected: vae
✅ Connected: clip_vision
✅ Connected: checkpoints
✅ Connected: loras

Model linking completed!


In [10]:
import subprocess
import sys
import time
import socket
from pathlib import Path

comfy = Path("/content/ComfyUI")
log_path = Path("/content/comfyui.log")

# Check whether server is already running
try:
    with socket.create_connection(("127.0.0.1", 8188), timeout=2):
        print("✅ ComfyUI is already running!")
except OSError:
    log = open(log_path, "w")

    proc = subprocess.Popen(
        [
            sys.executable,
            "main.py",
            "--listen", "127.0.0.1",
            "--port", "8188"
        ],
        cwd=str(comfy),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    print("Starting ComfyUI...")

    for i in range(90):
        if proc.poll() is not None:
            log.flush()
            print("❌ Server exited:")
            print(log_path.read_text()[-5000:])
            break

        try:
            with socket.create_connection(("127.0.0.1", 8188), timeout=2):
                print("✅ SUCCESS! ComfyUI is running on port 8188.")
                break
        except OSError:
            time.sleep(2)
    else:
        print("Server is taking longer than expected.")
        print(log_path.read_text()[-5000:])

Starting ComfyUI...
✅ SUCCESS! ComfyUI is running on port 8188.


In [11]:
!which cloudflared
!curl -sS -o /dev/null -w "ComfyUI HTTP status: %{http_code}\n" http://127.0.0.1:8188/

/usr/local/bin/cloudflared
ComfyUI HTTP status: 200


In [12]:
import subprocess
import time
import re
from pathlib import Path

log_path = Path("/content/cloudflared.log")
log = open(log_path, "w")

tunnel = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:8188",
        "--no-autoupdate"
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Starting Cloudflare tunnel...")

for i in range(60):
    time.sleep(1)
    log.flush()

    output = log_path.read_text(errors="replace")
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        output
    )

    if match:
        print("\nSUCCESS! Your ComfyUI link:")
        print(match.group(0))
        break

    if tunnel.poll() is not None:
        print("Tunnel exited unexpectedly:")
        print(output[-3000:])
        break
else:
    print("Tunnel URL not detected. Recent logs:")
    print(output[-3000:])

log.close()

Starting Cloudflare tunnel...

SUCCESS! Your ComfyUI link:
https://once-roy-atm-creating.trycloudflare.com


In [13]:
import subprocess
import sys
import time
import socket
from pathlib import Path

comfy = Path("/content/ComfyUI")
log_path = Path("/content/comfyui.log")

# Check whether server is already running
try:
    with socket.create_connection(("127.0.0.1", 8188), timeout=2):
        print("✅ ComfyUI is already running!")
except OSError:
    log = open(log_path, "w")

    proc = subprocess.Popen(
        [
            sys.executable,
            "main.py",
            "--listen", "127.0.0.1",
            "--port", "8188"
        ],
        cwd=str(comfy),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    print("Starting ComfyUI...")

    for i in range(90):
        if proc.poll() is not None:
            log.flush()
            print("❌ Server exited:")
            print(log_path.read_text()[-5000:])
            break

        try:
            with socket.create_connection(("127.0.0.1", 8188), timeout=2):
                print("✅ SUCCESS! ComfyUI is running on port 8188.")
                break
        except OSError:
            time.sleep(2)
    else:
        print("Server is taking longer than expected.")
        print(log_path.read_text()[-5000:])

✅ ComfyUI is already running!


In [14]:
import re
import shutil
import psutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

comfy = Path("/content/ComfyUI")
server_file = comfy / "server.py"

# 1. Patch ComfyUI's cross-site navigation check
source = server_file.read_text()

pattern = re.compile(
    r"(?m)^([ \t]*)if sec_fetch_site == ['\"]cross-site['\"]:\s*\n"
    r"\1[ \t]+return web\.Response\(status=403\)"
)

match = pattern.search(source)

if match:
    indent = match.group(1)

    replacement = (
        f"{indent}if sec_fetch_site == 'cross-site':\n"
        f"{indent}    is_top_level_navigation = (\n"
        f"{indent}        request.method == 'GET'\n"
        f"{indent}        and request.headers.get('Sec-Fetch-Mode') == 'navigate'\n"
        f"{indent}        and request.headers.get('Sec-Fetch-Dest') == 'document'\n"
        f"{indent}    )\n"
        f"{indent}    if not is_top_level_navigation:\n"
        f"{indent}        return web.Response(status=403)"
    )

    backup = Path("/content/server.py.before_navigation_fix")
    if not backup.exists():
        shutil.copy2(server_file, backup)

    source = source[:match.start()] + replacement + source[match.end():]
    server_file.write_text(source)
    print("✅ Navigation security patch applied.")

elif "is_top_level_navigation" in source:
    print("✅ Navigation patch already applied.")

else:
    raise RuntimeError("Expected security check not found. No restart performed.")

# 2. Stop only the existing ComfyUI server on port 8188
old_processes = []

for conn in psutil.net_connections(kind="inet"):
    if conn.status == psutil.CONN_LISTEN and conn.laddr.port == 8188 and conn.pid:
        try:
            p = psutil.Process(conn.pid)
            if p.pid != 0:
                old_processes.append(p)
        except psutil.Error:
            pass

for p in old_processes:
    try:
        p.terminate()
    except psutil.Error:
        pass

if old_processes:
    psutil.wait_procs(old_processes, timeout=10)
    print("✅ Previous ComfyUI server stopped.")

# 3. Restart ComfyUI
log = open("/content/comfyui.log", "w")

server_proc = subprocess.Popen(
    [
        sys.executable, "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188"
    ],
    cwd=str(comfy),
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Starting patched ComfyUI...")

# 4. Test normal and cross-site navigation requests
ready = False

for _ in range(90):
    if server_proc.poll() is not None:
        break

    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8188/",
            headers={
                "Sec-Fetch-Site": "cross-site",
                "Sec-Fetch-Mode": "navigate",
                "Sec-Fetch-Dest": "document"
            }
        )

        with urllib.request.urlopen(req, timeout=3) as response:
            print("✅ Cross-site navigation HTTP:", response.status)
            ready = response.status == 200
            break

    except Exception:
        time.sleep(2)

log.flush()

if ready:
    print("\n🎉 FIX SUCCESSFUL — ComfyUI is ready!")
else:
    print("\n❌ Startup/check failed. Recent log:")
    print(Path("/content/comfyui.log").read_text()[-5000:])

log.close()

✅ Navigation security patch applied.
✅ Previous ComfyUI server stopped.
Starting patched ComfyUI...

❌ Startup/check failed. Recent log:
/dist-packages/comfyui_frontend_package/static
[INFO] Asset seeder disabled
[INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'
[INFO] ### Loading: ComfyUI-Manager (V3.42)
[INFO] [ComfyUI-Manager] network_mode: public
[INFO] [ComfyUI-Manager] ComfyUI per-queue preview override detected (PR #11261). Manager's preview method feature is disabled. Use ComfyUI's --preview-method CLI option or 'Settings > Execution > Live preview method'.
[INFO] ### ComfyUI Version: v0.45.0-23-geee59185f | Released on '2026-11-13'
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/alter-list.json
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/model-list.json
[INFO] [ComfyUI-Manager] default cache updated: https://raw.github

In [15]:
import psutil
import subprocess
import sys
import time
import socket
from pathlib import Path

comfy = Path("/content/ComfyUI")
server_file = comfy / "server.py"

# Confirm our security patch is present
if "is_top_level_navigation" not in server_file.read_text():
    raise RuntimeError("Security patch is missing. Stop here.")

# Find only ComfyUI server processes
targets = []

for p in psutil.process_iter(["pid", "cmdline"]):
    try:
        cmd = p.info["cmdline"] or []
        cwd = Path(p.cwd()).resolve()

        if cwd == comfy.resolve() and any(
            x == "main.py" or x.endswith("/main.py") for x in cmd
        ):
            targets.append(p)

    except (psutil.NoSuchProcess, psutil.AccessDenied, FileNotFoundError):
        pass

print("Found ComfyUI processes:", [p.pid for p in targets])

# Stop old ComfyUI processes only
for p in targets:
    try:
        p.terminate()
    except psutil.NoSuchProcess:
        pass

psutil.wait_procs(targets, timeout=10)

for p in targets:
    try:
        if p.is_running():
            p.kill()
    except psutil.NoSuchProcess:
        pass

time.sleep(2)

# Make sure port is free
with socket.socket() as s:
    if s.connect_ex(("127.0.0.1", 8188)) == 0:
        raise RuntimeError("Port 8188 is still occupied. Stop here.")

# Start the patched ComfyUI server
log = open("/content/comfyui.log", "w")

server_proc = subprocess.Popen(
    [
        sys.executable,
        "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188"
    ],
    cwd=str(comfy),
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Starting patched ComfyUI...")

for i in range(90):
    if server_proc.poll() is not None:
        break

    try:
        request = __import__("urllib.request").request.Request(
            "http://127.0.0.1:8188/",
            headers={
                "Sec-Fetch-Site": "cross-site",
                "Sec-Fetch-Mode": "navigate",
                "Sec-Fetch-Dest": "document"
            }
        )

        with __import__("urllib.request").request.urlopen(
            request, timeout=3
        ) as response:
            print("Cross-site navigation HTTP:", response.status)
            break

    except Exception:
        time.sleep(2)
else:
    print("Server did not respond in time.")

log.close()

Found ComfyUI processes: [9576]
Starting patched ComfyUI...
Cross-site navigation HTTP: 200


In [16]:
import requests
from pathlib import Path

base = "http://127.0.0.1:8188"

print("COMFYUI API CHECK\n")

for endpoint in ["/", "/system_stats", "/extensions", "/object_info"]:
    try:
        r = requests.get(base + endpoint, timeout=15)
        print(f"{endpoint}: HTTP {r.status_code}")
    except Exception as e:
        print(f"{endpoint}: ERROR - {e}")

print("\nSECURITY PATCH CHECK")
source = Path("/content/ComfyUI/server.py").read_text()
print("Navigation patch present:", "is_top_level_navigation" in source)

print("\nRECENT SERVER LOG")
print(Path("/content/comfyui.log").read_text(errors="replace")[-2500:])

COMFYUI API CHECK

/: HTTP 200
/system_stats: HTTP 200
/extensions: HTTP 200
/object_info: HTTP 200

SECURITY PATCH CHECK
Navigation patch present: True

RECENT SERVER LOG
lib conf.c:5208:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5731:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5208:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5208:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5208:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5731:(snd_config_expand) Evaluate er

In [17]:
import requests

base = "http://127.0.0.1:8188"

for endpoint in ["/", "/system_stats", "/extensions", "/object_info"]:
    try:
        r = requests.get(base + endpoint, timeout=20)
        print(endpoint, "STATUS:", r.status_code,
              "SIZE:", len(r.content))
    except Exception as e:
        print(endpoint, "ERROR:", str(e))

/ STATUS: 200 SIZE: 19885
/system_stats STATUS: 200 SIZE: 1069
/extensions STATUS: 200 SIZE: 5561
/object_info STATUS: 200 SIZE: 3022955


In [18]:
from pathlib import Path

folders = [
    Path("/content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper/example_workflows"),
    Path("/content/ComfyUI/custom_nodes/ComfyUI-InfiniteTalk")
]

for folder in folders:
    print(f"\n--- {folder} ---")
    if folder.exists():
        for file in folder.rglob("*InfiniteTalk*.json"):
            print(file.name, f"({file.stat().st_size // 1024} KB)")
    else:
        print("Folder not found")


--- /content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper/example_workflows ---
wanvideo_2_1_14B_I2V_InfiniteTalk_example_03.json (47 KB)
wanvideo_2_1_14B_V2V_InfiniteTalk_example_02.json (55 KB)

--- /content/ComfyUI/custom_nodes/ComfyUI-InfiniteTalk ---


In [19]:
import json
import subprocess
from pathlib import Path
from google.colab import files

root = Path("/content/ComfyUI")
models = root / "models"
diffusion = models / "diffusion_models"

# Check existing downloaded models
required = {
    "Wan I2V": diffusion / "wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors",
    "UMT5": models / "text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "VAE": models / "vae/wan_2.1_vae.safetensors",
    "CLIP Vision": models / "clip_vision/clip_vision_h.safetensors",
}

for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
    print("✅", name)

# Place InfiniteTalk checkpoint in diffusion_models
multi_src = Path("/content/models/checkpoints/infinitetalk_multi_fp8.safetensors")
multi_dst = diffusion / multi_src.name

if not multi_src.exists():
    raise FileNotFoundError(multi_src)

if not multi_dst.exists():
    multi_dst.symlink_to(multi_src.resolve())

print("✅ InfiniteTalk checkpoint connected")

# Download the extra vocal separator model used by this workflow
mel = diffusion / "MelBandRoformer_fp16.safetensors"

if not mel.exists():
    url = (
        "https://huggingface.co/Kijai/MelBandRoFormer_comfy/"
        "resolve/main/MelBandRoformer_fp16.safetensors"
    )
    print("Downloading 456 MB vocal separator...")
    subprocess.run(
        ["wget", "-c", "-O", str(mel), url],
        check=True
    )

print("✅ Vocal separator ready")

# Load the official I2V workflow
workflow_path = (
    root / "custom_nodes/ComfyUI-WanVideoWrapper/"
    "example_workflows/"
    "wanvideo_2_1_14B_I2V_InfiniteTalk_example_03.json"
)

data = json.loads(workflow_path.read_text())
nodes = {n["id"]: n for n in data["nodes"]}

# Adapt model selections to our T4 FP8 setup
nodes[122]["widgets_values"][0] = required["Wan I2V"].name
nodes[122]["widgets_values"][4] = "sdpa"

nodes[120]["widgets_values"][0] = multi_dst.name

nodes[129]["widgets_values"][0] = required["VAE"].name

nodes[241]["widgets_values"][0] = required["UMT5"].name
nodes[241]["widgets_values"][4] = "fp8_e4m3fn"

# Disable the unavailable/unneeded example LoRA
nodes[138]["widgets_values"][1] = 0.0

# Vocal separator
nodes[301]["widgets_values"][0] = mel.name

# Landscape 480p dimensions
nodes[245]["widgets_values"][0] = 832
nodes[246]["widgets_values"][0] = 480

# Save customised workflow
output = Path("/content/InfiniteTalk_I2V_T4_FP8.json")
output.write_text(json.dumps(data, indent=2))

print("\n✅ Custom workflow created!")

# Download it to your Mac
files.download(str(output))

✅ Wan I2V
✅ UMT5
✅ VAE
✅ CLIP Vision
✅ InfiniteTalk checkpoint connected
✅ Vocal separator ready

✅ Custom workflow created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import subprocess
from pathlib import Path

root = Path("/content/ComfyUI/models")

# 1. InfiniteTalk checkpoint: reuse existing 18.16 GB file
name = "infinitetalk_multi_fp8.safetensors"
destination = root / "diffusion_models" / name

if not destination.exists():
    source = Path("/content/models/checkpoints") / name

    if not source.exists():
        raise FileNotFoundError(
            "Cannot find existing InfiniteTalk checkpoint. "
            "Stopping to avoid downloading another 19.5 GB."
        )

    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.symlink_to(source.resolve())

print("✅ InfiniteTalk checkpoint:", destination.exists())

# 2. Download the correct I2V LoRA
lora_name = "lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors"
lora = root / "loras/WanVideo/Lightx2v" / lora_name
lora.parent.mkdir(parents=True, exist_ok=True)

if not lora.exists():
    subprocess.run([
        "wget", "-c", "-O", str(lora),
        "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/" + lora_name
    ], check=True)

print("✅ I2V LoRA:", lora.exists())

# 3. Download Wav2Vec2
wav_name = "wav2vec2-chinese-base_fp16.safetensors"
wav = root / "wav2vec2" / wav_name
wav.parent.mkdir(parents=True, exist_ok=True)

if not wav.exists():
    subprocess.run([
        "wget", "-c", "-O", str(wav),
        "https://huggingface.co/Kijai/wav2vec2_safetensors/resolve/main/" + wav_name
    ], check=True)

print("✅ Wav2Vec2:", wav.exists())

print("\n🎉 Model file setup completed!")

✅ InfiniteTalk checkpoint: True
✅ I2V LoRA: True
✅ Wav2Vec2: True

🎉 Model file setup completed!


In [3]:
# 4. Download Models (FP8 for T4 Compatibility)
import os

# Define paths
models_dir = "models"
diffusion_dir = f"{models_dir}/diffusion_models"
text_enc_dir = f"{models_dir}/text_encoders"
vae_dir = f"{models_dir}/vae"
clip_dir = f"{models_dir}/clip_vision"
checkpoint_dir = f"{models_dir}/checkpoints"
loras_dir = f"{models_dir}/loras"

# Ensure directories exist
os.makedirs(diffusion_dir, exist_ok=True)
os.makedirs(text_enc_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(loras_dir, exist_ok=True)

print("Downloading Optimized FP8 Models...")

# Wan2.1 Diffusion Model (FP8 - 14B)
if not os.path.exists(f"{diffusion_dir}/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors"):
    !wget -c https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors -P {diffusion_dir}

# Text Encoder (UMT5 FP8)
if not os.path.exists(f"{text_enc_dir}/umt5_xxl_fp8_e4m3fn_scaled.safetensors"):
    !wget -c https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -P {text_enc_dir}

# VAE
if not os.path.exists(f"{vae_dir}/wan_2.1_vae.safetensors"):
    !wget -c https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -P {vae_dir}

# CLIP Vision
if not os.path.exists(f"{clip_dir}/clip_vision_h.safetensors"):
    !wget -c https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors -P {clip_dir}

# Wan2.1 LoRA (Kijai)
if not os.path.exists(f"{loras_dir}/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors"):
    !wget -c https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors -P {loras_dir}

# InfiniteTalk Checkpoints (MeiGen)
if not os.path.exists(f"{checkpoint_dir}/infinitetalk_multi_fp8.safetensors"):
    !wget -c https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/infinitetalk_multi_fp8.safetensors -P {checkpoint_dir}

print("✅ All Models Downloaded!")

--2026-11-14 02:16:56--  https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.118, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /Comfy-Org/Wan-2.1/resolve/main/split_files/diffusion_models/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors [following]
--2026-11-14 02:16:56--  https://huggingface.co/Comfy-Org/Wan-2.1/resolve/main/split_files/diffusion_models/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/67be35b066f702bfed7d3bdc/f480e31ff1021f112f0f367d08440b6355e4db1a63de8ea991fcac8b9a0ba27f?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wan2.1_i2v_480

In [4]:
import subprocess
import sys
import time
import socket
from pathlib import Path

comfy = Path("/content/ComfyUI")

if not (comfy / "main.py").exists():
    print("ComfyUI installation not found!")
else:
    log = open("/content/comfyui.log", "w")

    proc = subprocess.Popen(
        [
            sys.executable,
            "main.py",
            "--listen", "127.0.0.1",
            "--port", "8188"
        ],
        cwd=str(comfy),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    print("Starting ComfyUI...")

    for i in range(90):
        if proc.poll() is not None:
            log.flush()
            print("Startup failed:")
            print(Path("/content/comfyui.log").read_text()[-5000:])
            break

        try:
            with socket.create_connection(
                ("127.0.0.1", 8188), timeout=2
            ):
                print("SUCCESS! ComfyUI is running on port 8188.")
                break
        except OSError:
            time.sleep(2)
    else:
        print("Still starting. Check /content/comfyui.log")

ComfyUI installation not found!


In [1]:
# 5. Install Cloudflared & Launch
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("🚀 Starting ComfyUI...")
print("Look for the trycloudflare.com URL below to access the interface.\t")

# Run ComfyUI in background, expose via tunnel
!python main.py --listen --port 8188 & cloudflared tunnel --url http://localhost:8188

--2026-11-14 02:09:02--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64.deb [following]
--2026-11-14 02:09:02--  https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/04372d98-d20f-4427-9a16-58016e29739c?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-14T03%3A04%3A33Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4d

In [2]:
import os
from pathlib import Path

print("CURRENT DIRECTORY:", os.getcwd())
print("\nCONTENT DIRECTORY:")

if Path("/content").exists():
    print(os.listdir("/content"))

print("\nSEARCHING FOR COMFYUI:")
for p in Path("/content").rglob("main.py"):
    if p.parent.name == "ComfyUI" or "ComfyUI" in str(p):
        print(p)

CURRENT DIRECTORY: /content

CONTENT DIRECTORY:
['.config', 'cloudflared-linux-amd64.deb', 'sample_data']

SEARCHING FOR COMFYUI:
